# Quickstart analyse GPS

Ce notebook montre le chemin le plus court pour un analyste : charger le test set anonymisé, préparer les tables de mobilité, calculer quelques indicateurs et produire une première figure.

Le calcul est volontairement limité à 4 utilisateurs pour garder un temps d'exécution raisonnable. Le notebook fil rouge couvre ensuite le pipeline complet.

## 1. Préparer l'environnement

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)

# En développement local, on importe le code source du dépôt.
# Après publication, `pip install xyt-gps` suffira et ce bloc n'aura aucun effet.
for root in (Path.cwd(), *Path.cwd().parents):
    candidates = [root / "package-xyt-gps" / "src", root / "src"]
    for candidate in candidates:
        if (candidate / "xyt_gps").exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

import xyt_gps as xyt


def find_demo_data_dir() -> Path:
    env_path = os.environ.get("XYT_GPS_DEMO_DATA_DIR")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    for root in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([
            root / "Data" / "Output" / "anonymous-test-set-gps",
            root / "anonymous-test-set-gps",
        ])
    for candidate in candidates:
        if (candidate / "storyline.csv").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Jeu de démonstration introuvable. "
        "Définir XYT_GPS_DEMO_DATA_DIR ou demander le dossier anonymous-test-set-gps à contact@situee.ch."
    )


DATA_DIR = find_demo_data_dir()
pd.Series({"data_dir": DATA_DIR.name}, name="environnement")

## 2. Charger un petit échantillon

Le test set contient 12 utilisateurs. Pour un quickstart, on garde 4 utilisateurs complets, puis on filtre toutes les tables sur le même périmètre.

In [ ]:
TABLE_NAMES = [
    "storyline",
    "trips",
    "journeys",
    "user_statistics",
    "sociodemographics",
]

tables = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in TABLE_NAMES}
selected_users = sorted(tables["storyline"]["user_id"].dropna().astype(str).unique())[:4]

sample_tables = {}
for name, table in tables.items():
    if "user_id" in table.columns:
        sample_tables[name] = table.loc[table["user_id"].astype(str).isin(selected_users)].copy()
    else:
        sample_tables[name] = table.copy()

volume = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
        }
        for name, table in sample_tables.items()
    ]
)

display(pd.Series({"data_dir": DATA_DIR.name, "selected_users": len(selected_users)}, name="contexte"))
display(volume)

## 3. Préparer les tables et les indicateurs

`ProjectConfig` rend explicites le périmètre temporel, les phases et les paramètres utiles. Ici les options lourdes de qualité spatiale sont désactivées pour accélérer le quickstart.

In [ ]:
phases = (
    xyt.Phase("Phase1", "2025-04-14", "2025-05-04"),
    xyt.Phase("Phase2", "2025-05-05", "2025-06-01"),
    xyt.Phase("Phase3", "2025-06-02", "2025-06-15"),
)

config = xyt.ProjectConfig(
    experiment_name="anonymous-test-set-gps",
    motiontag_project_name="test-project-gps",
    start_expe="2025-04-14",
    end_expe="2025-06-15",
    phases=phases,
    reference_year=2025,
)

raw = xyt.RawGpsData(
    storyline=sample_tables["storyline"],
    trips=sample_tables["trips"],
    journeys=sample_tables["journeys"],
    user_statistics=sample_tables["user_statistics"],
)

result = xyt.run_mobility_pipeline(
    config,
    raw=raw,
    sociodemo=sample_tables["sociodemographics"],
    resample_missing_days=False,
    clean_leg_geometries=False,
    add_length_outlier_flags=False,
    add_signal_quality_flags=False,
)

dataset = result.dataset
indicators = result.indicators

## 4. Lire les résultats

In [ ]:
tables_prepared = xyt.mobility_dataset_tables(dataset)
prepared_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
        }
        for name, table in tables_prepared.items()
    ]
)

indicator_view = (
    indicators.population
    .sort_values(["phase", "distance_km_per_day"], ascending=[True, False])
    .round({"distance_km_per_day": 1, "travel_time_min_per_day": 1, "trip_count_per_day": 2})
)

display(prepared_summary)
display(indicator_view.head(12))

## 5. Produire une première figure

In [ ]:
indicator_fig = xyt.plot_indicator_bars(
    indicators,
    metrics=["distance_km_per_day", "travel_time_min_per_day"],
    max_bars=6,
    show_demand_profile=False,
    title="Quickstart : distance et temps de trajet par mode",
)

display(indicator_fig)

## Points à retenir

- `RawGpsData` regroupe les tables d'entrée : `storyline`, `trips`, `journeys` et `user_statistics`.
- `ProjectConfig` garde les choix méthodologiques visibles : dates, phases, CRS, mappings et seuils.
- `run_mobility_pipeline()` est l'entrée la plus courte pour passer des tables GPS aux indicateurs.
- Pour un traitement complet, utiliser ensuite `demo-fil-rouge-pipeline.ipynb`.